# Practice 2 · Challenge B (storage)

**R3 · Regression lead** · working notebook (`notebooks/borradores/R3_storage.ipynb`)

In this notebook we do the parts of Challenge B that belong to R3:

- **P2.3:** we look at the target, fit a baseline and a KNN on the raw
  `latency_ns` and on $z = \log_{10}(\texttt{latency\_ns})$, and justify why we use $z$.
- **P2.4:** baseline and KNN on $z$, with a grid of $K_{NN}$.
- the storage part of the **P2.6** dry run.

All scores use the canonical folds of `storage_folds.csv` (P2.2 g). We only read
them, we don't build them again.

**Notation.** $K = 5$ is the number of folds and $K_{NN}$ the number of
neighbours. In the code the neighbour count is always `n_neighbors` or `K_NN`.

# 0. Setup

In [ ]:
PROJECT_RANDOM_STATE = 2627
print(f"PROJECT_RANDOM_STATE = {PROJECT_RANDOM_STATE}")

In [ ]:
%matplotlib inline

# --- Imports ---
from pathlib import Path   # to build file paths that work on Windows and Linux

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

import sklearn

# Library versions, so anyone re-running the notebook can compare
print("numpy       ", np.__version__)
print("pandas      ", pd.__version__)
print("matplotlib  ", matplotlib.__version__)
print("scikit-learn", sklearn.__version__)

### Constants

We fix the paths and the role of each column before reading any data. The lists
come from the data contract (statement and `PROVENANCE.md`) and are checked
against the files in the next section.

`raid` goes with the nominal columns: `4+2` is a RAID scheme, not a number, so it
is one-hot encoded.

In [ ]:
# --- Paths, relative to notebooks/borradores/ ---
DATA_DIR = Path("../../datos/storage")

DEVELOPMENT_PATH = DATA_DIR / "storage_development.csv"
GROUPS_PATH = DATA_DIR / "storage_groups.csv"
FOLDS_PATH = DATA_DIR / "storage_folds.csv"     # canonical, course-owned (P2.2 g)

# --- Column roles ---
ID_COLUMN = "row_id"            # identifier only, never a predictor
TARGET = "latency_ns"           # target as distributed (we model z = log10 of it)
GROUP_COLUMN = "config_group"   # evaluation metadata only, never an estimator input
FOLD_COLUMN = "fold_id"         # evaluation metadata only, never an estimator input

# The 9 predictors of the contract: 5 numeric and 4 nominal (raid is nominal: "4+2" is a scheme)
NUMERIC_FEATURES = ["block_size", "n_jobs", "iodepth", "read_fraction", "n_disks"]
NOMINAL_FEATURES = ["load_type", "io_type", "raid", "device_type"]
FEATURES = NUMERIC_FEATURES + NOMINAL_FEATURES

CV_FOLDS = 5          # K, the number of cross-validation folds
REPLICATES = 5        # replicate measurements per configuration (PROVENANCE.md)

## 1. Loading the three tables

We load the development table, the groups and the canonical folds, check their
columns against the contract and join them by `row_id`. The join is checked as
one-to-one, so a missing or repeated id would stop the notebook here.

The full P2.1 exploration is R1's part; here we only check what we use later.

In [ ]:
# --- Load the three tables ---
development = pd.read_csv(DEVELOPMENT_PATH)   # row_id, 9 predictors, latency_ns
groups = pd.read_csv(GROUPS_PATH)             # row_id, config_group
folds = pd.read_csv(FOLDS_PATH)               # row_id, fold_id

# 1. Schema against the contract
# Same columns as the contract (the file stores them in its own order).
assert development.columns[0] == ID_COLUMN and development.columns[-1] == TARGET
assert set(development.columns) == {ID_COLUMN, TARGET, *FEATURES}
assert len(development.columns) == len(FEATURES) + 2
assert list(groups.columns) == [ID_COLUMN, GROUP_COLUMN]
assert list(folds.columns) == [ID_COLUMN, FOLD_COLUMN]
# iops is a second output of the same run, so it must not be in the table (rule 4)
assert "iops" not in development.columns

# 2. Join the three tables by row_id
# validate="one_to_one" raises an error if a row_id is missing or repeated in any table
storage = (development
           .merge(groups, on=ID_COLUMN, how="inner", validate="one_to_one")
           .merge(folds, on=ID_COLUMN, how="inner", validate="one_to_one"))

# No row lost in the join
assert len(storage) == len(development) == len(groups) == len(folds)

# 3. Quick summary of what we loaded
print(f"development rows        {len(development)}")
print(f"rows after the join     {len(storage)}")
print(f"source predictors       {len(FEATURES)}  ({len(NUMERIC_FEATURES)} numeric, "
      f"{len(NOMINAL_FEATURES)} nominal)")
print(f"distinct config_group   {storage[GROUP_COLUMN].nunique()}")
print(f"missing values          {int(storage[FEATURES + [TARGET]].isna().to_numpy().sum())}")
print(f"latency_ns all > 0      {bool((storage[TARGET] > 0).all())}")

### Canonical folds (P2.2 g)

The folds belong to the course, so we only read them. We check again that the 5
replicates of each configuration are always in the same fold, because every
result below depends on it (the full check is in R4's P2.2).

In [ ]:
# --- Canonical folds: size of each fold ---
# For each fold_id: how many rows and how many configurations are validated in it
fold_table = (storage.groupby(FOLD_COLUMN)
              .agg(validation_rows=(ID_COLUMN, "size"),
                   configurations=(GROUP_COLUMN, "nunique"))
              .reset_index())
# The training part of a fold is every row that is not in its validation part
fold_table["training rows"] = len(storage) - fold_table["validation_rows"]
fold_table = fold_table.rename(columns={FOLD_COLUMN: "fold", "validation_rows": "validation rows"})
print(fold_table.to_string(index=False))
print()

# --- Grouping invariant ---
# Number of different folds each configuration appears in (must be 1)
folds_per_config = storage.groupby(GROUP_COLUMN)[FOLD_COLUMN].nunique()
# Number of rows of each configuration (must be the 5 replicates)
rows_per_config = storage.groupby(GROUP_COLUMN).size()
print(f"configurations split across folds   {int((folds_per_config > 1).sum())}")
print(f"replicates per configuration        {sorted(rows_per_config.unique())}")

assert sorted(storage[FOLD_COLUMN].unique()) == list(range(CV_FOLDS))   # folds 0 to 4
assert (folds_per_config == 1).all()        # no configuration split across folds
assert (rows_per_config == REPLICATES).all()

## 2. P2.3: distribution of `latency_ns`

Before fitting anything we look at the target as it is given, and we compute the
same summaries on $\log_{10}$ to compare. Nothing is fitted in this section.

In [ ]:
# --- Summary of the target, raw and on the log10 scale ---
latency = storage[TARGET].to_numpy()
log_latency = np.log10(latency)

def describe(values):
    # Main summaries of a 1-D array; skewness > 0 means a long right tail
    return {"min": values.min(), "median": np.median(values), "mean": values.mean(),
            "max": values.max(), "skewness": pd.Series(values).skew()}

# One row per scale, so both can be read side by side
summary = pd.DataFrame({"latency_ns (raw)": describe(latency),
                        "log10(latency_ns)": describe(log_latency)}).T
print(summary.to_string(float_format=lambda v: f"{v:12.4g}"))
print()

# How wide and how skewed the raw target is
span = latency.max() / latency.min()
print(f"max / min                          {span:,.0f}")
print(f"orders of magnitude covered        {np.log10(span):.2f}")   # log10 of the ratio
print(f"mean / median (raw)                {latency.mean() / np.median(latency):.1f}")
print(f"rows below the raw mean            {(latency < latency.mean()).mean():.1%}")
# Sort the latencies, keep the last 1% (the slowest rows) and compare with the total
top_share = np.sort(latency)[-len(latency) // 100:].sum() / latency.sum()
print(f"share of the total latency held by the slowest 1% of rows   {top_share:.1%}")

In [ ]:
# --- Histograms of the target: raw (left) and log10 (right) ---
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))

# Left: raw latency, with its median and its mean marked
axes[0].hist(latency, bins=80, color="#B0BEC5", edgecolor="#023E83", linewidth=0.4)
axes[0].axvline(np.median(latency), color="#1E7F4E", linestyle="--", linewidth=1.4,
                label=f"median {np.median(latency):.3g} ns")
axes[0].axvline(latency.mean(), color="#B00020", linestyle=":", linewidth=1.6,
                label=f"mean {latency.mean():.3g} ns")
axes[0].set_xlabel("latency_ns")
axes[0].set_ylabel("rows")
axes[0].set_title("Raw target")
axes[0].legend(fontsize=8.5)

# Right: the same rows after log10
axes[1].hist(log_latency, bins=80, color="#DCE7F4", edgecolor="#023E83", linewidth=0.4)
axes[1].axvline(np.median(log_latency), color="#1E7F4E", linestyle="--", linewidth=1.4,
                label="median")
axes[1].axvline(log_latency.mean(), color="#B00020", linestyle=":", linewidth=1.6,
                label="mean")
axes[1].set_xlabel("z = log10(latency_ns)")
axes[1].set_ylabel("rows")
axes[1].set_title("The same rows on a log10 scale")
axes[1].legend(fontsize=8.5)

for ax in axes:
    ax.spines[["top", "right"]].set_visible(False)   # remove the top and right borders
fig.suptitle(f"Development target, n = {len(latency)} rows", y=1.02)
plt.tight_layout()
plt.show()

**What we see.** On the raw scale almost every row falls in the first bar of the
histogram. The target goes from about $1.2\times10^5$ to $2.5\times10^9$ ns (4.3
orders of magnitude) and is very skewed (skewness ≈ 9): the mean is 23 times the
median, 86% of the rows are below the mean, and the slowest 1% of rows add up to
almost a third of all the latency in the table.

On the log scale the same rows fit in about 4 units and the shape is visible: a
peak around $z \approx 5.8$ (below one millisecond) and a long, flatter tail up to
$z \approx 9$. The skewness drops to about 1. The log does not make the target
normal, and we don't need it to.

### Where the orders of magnitude come from

The latency changes a lot with the configuration. The table groups the rows by
`device_type` and `load_type`.

In [ ]:
# --- Latency by device and access pattern ---
# For each (device_type, load_type) combination: number of rows, median, min and max latency
by_device = (storage.groupby(["device_type", "load_type"])[TARGET]
             .agg(rows="size", median="median", minimum="min", maximum="max")
             .reset_index())
# How many times larger the slowest row is than the fastest one, inside each combination
by_device["max / min"] = by_device["maximum"] / by_device["minimum"]
print(by_device.to_string(index=False, float_format=lambda v: f"{v:12.4g}"))

### How the replicate noise changes with the latency

Each configuration was measured 5 times, so the spread of its 5 replicates shows
how noisy the measurement is. If the noise were additive, the spread in ns would be
similar at every latency level. If it is multiplicative (a similar percentage),
the spread in ns grows with the latency and the spread in $z$ stays roughly flat.
We split the configurations into five levels by their mean $z$.

In [ ]:
# --- Replicate noise per configuration ---
# 1. One row per configuration: mean and standard deviation of its 5 replicates, in ns and in z
per_config = storage.assign(z=log_latency).groupby(GROUP_COLUMN).agg(
    mean_ns=(TARGET, "mean"), sd_ns=(TARGET, "std"),
    mean_z=("z", "mean"), sd_z=("z", "std"))
# Coefficient of variation: the sd as a fraction of the mean (relative noise)
per_config["cv_ns"] = per_config["sd_ns"] / per_config["mean_ns"]

# 2. Configurations grouped by latency level (quintiles of their mean z), so the
# noise can be read at each level.
per_config["level"] = pd.qcut(per_config["mean_z"], 5,
                              labels=["fastest 20%", "2nd", "3rd", "4th", "slowest 20%"])
# 3. Typical (median) noise in each level
noise_table = (per_config.groupby("level", observed=True)
               .agg(configurations=("sd_ns", "size"),
                    median_latency_ns=("mean_ns", "median"),
                    median_sd_ns=("sd_ns", "median"),
                    median_cv=("cv_ns", "median"),
                    median_sd_z=("sd_z", "median"))
               .reset_index())
print(noise_table.to_string(index=False, float_format=lambda v: f"{v:12.4g}"))
print()
# 4. How much the noise grows from the fastest level to the slowest one, in each scale
ratio_sd_ns = noise_table["median_sd_ns"].iloc[-1] / noise_table["median_sd_ns"].iloc[0]
ratio_sd_z = noise_table["median_sd_z"].iloc[-1] / noise_table["median_sd_z"].iloc[0]
print(f"replicate sd in ns, slowest / fastest quintile   {ratio_sd_ns:,.0f}x")
print(f"replicate sd of z,  slowest / fastest quintile   {ratio_sd_z:.1f}x")

In [ ]:
# --- Noise against level, one point per configuration ---
fig, axes = plt.subplots(1, 2, figsize=(11, 3.9))

# Left: in ns. Both axes in log scale because the values cover several orders of magnitude
axes[0].scatter(per_config["mean_ns"], per_config["sd_ns"], s=6, alpha=0.35,
                color="#023E83")
axes[0].set_xscale("log")
axes[0].set_yscale("log")
axes[0].set_xlabel("configuration mean latency_ns (log axis)")
axes[0].set_ylabel("replicate sd in ns (log axis)")
axes[0].set_title("Raw scale: the noise grows with the level")

# Right: the same configurations measured in z
axes[1].scatter(per_config["mean_z"], per_config["sd_z"], s=6, alpha=0.35,
                color="#1E7F4E")
axes[1].set_xlabel("configuration mean z")
axes[1].set_ylabel("replicate sd of z")
axes[1].set_title("log10 scale: far weaker dependence on the level")

for ax in axes:
    ax.spines[["top", "right"]].set_visible(False)
fig.suptitle(f"One point per configuration ({len(per_config)} configurations, "
             f"{REPLICATES} replicates each)", y=1.02)
plt.tight_layout()
plt.show()

**What we see.**

1. **The range depends on the configuration.** The three combinations in the data
   (there is no `hdd` + `random`) have medians up to two orders of magnitude
   apart, and each one still covers about three orders of magnitude. The
   predictors seem to multiply the latency (slower device, bigger blocks, deeper
   queue) rather than add a fixed number of ns.
2. **The noise is mostly multiplicative.** From the fastest to the slowest level,
   the replicate standard deviation grows about 1,300 times in ns but only about
   9 times in $z$. So the log removes most of the dependence on the level, but not
   all of it: the slowest configurations are also relatively noisier (coefficient
   of variation around 11% against about 1%).

These are only observations. The argument for the target comes after the fits of
section 4.

## 3. Code from R1 and R4

From here we need three pieces of team code. Following the team rules, they are
copied **without changes** from their notebooks. If something has to change, the
owner changes it and we copy it again.

| Piece | Owner | Source | What it does |
|---|---|---|---|
| `make_storage_preprocessor()` | R1 | `R1_datos.ipynb`, section 4 | unfitted `ColumnTransformer`: one-hot on the 4 nominal predictors, `StandardScaler` on the 5 numeric ones |
| `canonical_cv("storage")` | R4 | `R4_evaluacion.ipynb`, section 9 | `PredefinedSplit` built from `storage_folds.csv` |
| `evaluate_storage(...)` | R4 | `R4_evaluacion.ipynb`, section 9 | refits a copy of the model on every canonical fold; MAE, RMSE and $R^2$ on $z$, and MAE in ns after $10^{\hat z}$ |

R1's function uses the names `STORAGE_NOMINAL` and `STORAGE_NUMERIC`, so we point
those names to our own lists first.

In [ ]:
# Names expected by R1's function, bound to the lists declared in the constants cell
STORAGE_NOMINAL = NOMINAL_FEATURES
STORAGE_NUMERIC = NUMERIC_FEATURES

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# --- Copied unchanged from R1_datos.ipynb, section 4 (preprocessing block, storage) ---
def make_storage_preprocessor():
    # Nominal (raid included): one column per category; unseen categories become all zeros
    nominal_steps = OneHotEncoder(handle_unknown="ignore")

    # Numeric: put every column on the same scale
    numeric_steps = StandardScaler()

    preprocessor = ColumnTransformer([
        ("nominal", nominal_steps, STORAGE_NOMINAL),
        ("numeric", numeric_steps, STORAGE_NUMERIC),
    ])
    return preprocessor

In [ ]:
# --- Copied unchanged from R4_evaluacion.ipynb, section 9 (storage parts only) ---
from sklearn.model_selection import PredefinedSplit

def canonical_cv(challenge):
    """Canonical 5-fold cv for cross_validate: 'cyber' or 'storage'.

    Reads the course-owned fold_id file; fold_id is never an estimator input.
    """
    path = f"../../datos/{challenge}/{challenge}_folds.csv"
    return PredefinedSplit(test_fold=pd.read_csv(path)["fold_id"].to_numpy())


import time

from sklearn.base import clone
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


def evaluate_storage(estimator, X, z, latency_ns, cv):
    """Challenge B under the metric contract and the log-to-physical rule, on `cv`.

    MAE(z), RMSE(z) and R2(z) are computed directly from z and z_hat. MAE_ns is computed
    only after inverting the prediction (10**z_hat) and against the raw latency_ns;
    10**MAE(z) is never reported as an error in nanoseconds.
    """
    z = np.asarray(z)
    latency_ns = np.asarray(latency_ns)
    rows = []
    for k, (tr, va) in enumerate(cv.split(X, z)):
        model = clone(estimator)
        t0 = time.perf_counter()
        model.fit(X.iloc[tr], z[tr])
        fit_time = time.perf_counter() - t0
        z_hat = model.predict(X.iloc[va])
        rows.append({"fold": k + 1,
                     "mae_z": mean_absolute_error(z[va], z_hat),
                     "rmse_z": np.sqrt(mean_squared_error(z[va], z_hat)),
                     "r2_z": r2_score(z[va], z_hat),
                     "mae_ns": mean_absolute_error(latency_ns[va], 10 ** z_hat),
                     "fit_time_s": fit_time})
    return {"fold_metrics": pd.DataFrame(rows).set_index("fold")}

### Modelling arrays and canonical `cv`

`X` only contains the nine predictors. `row_id`, `config_group` and `fold_id` stay
in the table but never go into `.fit()`.

`PredefinedSplit` works by position, so we check that the rows are in the same
order as `storage_folds.csv` and that each validation fold is exactly the set of
rows with that `fold_id`.

In [ ]:
# --- Modelling arrays ---
# X: only the 9 predictors (no row_id, config_group or fold_id)
X = storage[FEATURES]
latency_ns = storage[TARGET].to_numpy()
z = np.log10(latency_ns)            # z = log10(latency_ns), the representation under study

# Canonical folds as a cv object for scikit-learn (R4's function)
cv = canonical_cv("storage")

# --- Checks ---
# 1. PredefinedSplit uses row positions, so our rows must be in the order of the fold file
assert storage[ID_COLUMN].equals(folds[ID_COLUMN])            # same row order as the fold file
# 2. No identifier, group, fold or target column among the inputs
assert not set(X.columns) & {ID_COLUMN, GROUP_COLUMN, FOLD_COLUMN, TARGET}
# 3. For each fold: the validation rows are exactly the rows with that fold_id,
#    and no configuration is both in training and in validation
for q, (train_idx, valid_idx) in enumerate(cv.split()):
    assert set(valid_idx) == set(np.flatnonzero(storage[FOLD_COLUMN].to_numpy() == q))
    assert not set(storage[GROUP_COLUMN].iloc[train_idx]) & set(storage[GROUP_COLUMN].iloc[valid_idx])

print(f"X shape                     {X.shape}")
print(f"estimator inputs            {list(X.columns)}")
print(f"canonical folds             {cv.get_n_splits()}")
print(f"configurations shared between a fold's training and validation rows: 0 (asserted)")

## 4. P2.3: baseline and KNN on the raw target and on $z$

The statement asks for exactly two fits on the raw target and the same two on $z$,
with no search on the raw target. We fix both models before looking at any score:

- **baseline:** `DummyRegressor(strategy="mean")`, which always predicts the
  training mean and ignores `X`;
- **KNN:** R1's preprocessing + `KNeighborsRegressor(n_neighbors=5)`. We don't
  choose $K_{NN} = 5$ here: it is the fixed configuration of P2.2, so the only thing
  that changes between the runs is the target. $K_{NN}$ is chosen in P2.4, only
  on $z$.

In [ ]:
from sklearn.dummy import DummyRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_predict

P23_NEIGHBOURS = 5     # course-fixed comparison configuration (P2.2), not tuned


def make_baseline():
    # Always predicts the mean of the training target; it never looks at X
    return DummyRegressor(strategy="mean")


def make_knn(n_neighbours):
    # R1's preprocessing + KNN inside one Pipeline, so the encoder and the scaler
    # are fitted only on the training rows of each fold
    return Pipeline([
        ("prep", make_storage_preprocessor()),
        ("knn", KNeighborsRegressor(n_neighbors=n_neighbours)),
    ])

### Scoring a model fitted on the raw target

R4's `evaluate_storage` always fits on $z$ and then computes $10^{\hat z}$. That is
right for the rest of the project, but not for the two raw-target fits. For those
two we wrote `evaluate_storage_raw`. It does the same steps (copy the model, fit it
on the training folds, predict the validation fold) but fits on `latency_ns`, so:

- `mae_ns` is computed directly, because the prediction is already in ns;
- `mae_z`, `rmse_z` and `r2_z` are computed on $\log_{10}$ of the prediction, so we
  can compare the four models with the primary metric.

The predictions are averages of positive latencies, so the log is always defined
(the function checks it anyway). It also returns the out-of-fold predictions,
which we use in section 4.2.

In [ ]:
# R3's own helper, used ONLY for the two raw-target fits of P2.3.
# Same structure as R4's evaluate_storage, but the model is fitted on latency_ns.
def evaluate_storage_raw(estimator, X, latency_ns, cv):
    """Fit on raw latency_ns in every canonical fold; same metric columns as evaluate_storage."""
    latency_ns = np.asarray(latency_ns)
    z_true = np.log10(latency_ns)                      # true z, to compute the log metrics
    oof_pred_ns = np.full(len(latency_ns), np.nan)     # out-of-fold predictions, filled fold by fold
    rows = []
    for k, (tr, va) in enumerate(cv.split(X, latency_ns)):
        # 1. Fresh copy of the model for this fold, fitted on the training rows only
        model = clone(estimator)
        t0 = time.perf_counter()
        model.fit(X.iloc[tr], latency_ns[tr])
        fit_time = time.perf_counter() - t0
        # 2. Predict the validation rows: the prediction is already in ns
        pred_ns = model.predict(X.iloc[va])
        assert (pred_ns > 0).all()           # log10 of the prediction must be defined
        oof_pred_ns[va] = pred_ns
        # 3. Metrics: on z after log10 of the prediction, and MAE directly in ns
        z_hat = np.log10(pred_ns)
        rows.append({"fold": k + 1,
                     "mae_z": mean_absolute_error(z_true[va], z_hat),
                     "rmse_z": np.sqrt(mean_squared_error(z_true[va], z_hat)),
                     "r2_z": r2_score(z_true[va], z_hat),
                     "mae_ns": mean_absolute_error(latency_ns[va], pred_ns),
                     "fit_time_s": fit_time})
    return {"fold_metrics": pd.DataFrame(rows).set_index("fold"), "oof_pred_ns": oof_pred_ns}

### The four fits

The two raw-target models use `evaluate_storage_raw` and the two $z$ models use
R4's `evaluate_storage`. For the $z$ models we also get the out-of-fold
predictions with `cross_val_predict` on the same folds, only for section 4.2. The
cell checks that they give exactly R4's MAE(z) in every fold, so they are the same
models and not new ones.

In [ ]:
# --- The four fits of P2.3 ---
# Raw target -> our helper; z -> R4's evaluate_storage (unchanged)
results = {
    "Dummy, raw target": evaluate_storage_raw(make_baseline(), X, latency_ns, cv),
    f"KNN (K_NN={P23_NEIGHBOURS}), raw target": evaluate_storage_raw(make_knn(P23_NEIGHBOURS), X, latency_ns, cv),
    "Dummy, z": evaluate_storage(make_baseline(), X, z, latency_ns, cv),
    f"KNN (K_NN={P23_NEIGHBOURS}), z": evaluate_storage(make_knn(P23_NEIGHBOURS), X, z, latency_ns, cv),
}

# Out-of-fold predictions in ns for every model (diagnostics of section 4.2 only)
# The raw-target models already returned them
oof_ns = {name: res["oof_pred_ns"] for name, res in results.items() if "oof_pred_ns" in res}
# For the z models we get them with cross_val_predict on the same folds
for name, estimator in [("Dummy, z", make_baseline()),
                        (f"KNN (K_NN={P23_NEIGHBOURS}), z", make_knn(P23_NEIGHBOURS))]:
    z_oof = cross_val_predict(estimator, X, z, cv=cv)
    # Check: they must give exactly R4's MAE(z) in every fold (same models, not new ones)
    for q, (_, va) in enumerate(cv.split(), start=1):
        assert abs(mean_absolute_error(z[va], z_oof[va])
                   - results[name]["fold_metrics"].loc[q, "mae_z"]) < 1e-12
    oof_ns[name] = 10 ** z_oof     # back to ns

# --- Comparison table: MAE(z) per fold, mean, range and the other metrics ---
table = []
for name, res in results.items():
    fm = res["fold_metrics"]
    table.append({"model": name,
                  **{f"MAE(z) f{q}": round(fm.loc[q, "mae_z"], 4) for q in fm.index},
                  "MAE(z) mean": round(fm["mae_z"].mean(), 4),
                  "MAE(z) range": round(fm["mae_z"].max() - fm["mae_z"].min(), 4),
                  "RMSE(z)": round(fm["rmse_z"].mean(), 4),
                  "R2(z)": round(fm["r2_z"].mean(), 4),
                  "MAE_ns": f"{fm['mae_ns'].mean():.3e}"})
p23_table = pd.DataFrame(table)
print(p23_table.to_string(index=False))

**What we see** (means over the 5 canonical folds):

- **The baselines are very different.** The raw-target baseline has MAE(z) = 1.231
  against 0.677 for the $z$ baseline, and $R^2(z) \approx -1.6$, so on the log
  scale it is worse than a constant. It is also worse in ns (MAE$_\text{ns}$ about
  $4.2\times10^7$ against $2.7\times10^7$). Section 4.2 explains why.
- **The two KNNs are almost the same.** MAE(z) 0.2103 (raw) against 0.2091 ($z$).
  The per-fold differences (raw − $z$) are 0.0011, 0.0023, 0.0006, 0.0016 and
  0.0002: same sign in all folds, but much smaller than the fold-to-fold range of
  each model (about 0.012). This only describes these five folds; it is not a
  significance test.

So the KNN scores don't decide which target to use, and the justification in 4.3
does not depend on them.

### 4.2 What changes with the representation

From the table alone we could say "the log is better because it scores better".
Instead, we look at what each model predicts and at where each error measure puts
its weight, using the out-of-fold predictions.

**(a) Baseline.** The Dummy predicts the mean of the target it receives. On the raw
target that is the arithmetic mean of the latencies. On $z$ it is the mean of the
logs, which in ns is the **geometric** mean, $10^{\bar z}$.

In [ ]:
# --- (a) What each baseline predicts ---
# Each baseline predicts one constant per fold; the median of its predictions is that constant
dummy_raw_value = float(np.median(oof_ns["Dummy, raw target"]))   # one constant per fold
dummy_z_value = float(np.median(oof_ns["Dummy, z"]))
median_latency = float(np.median(latency_ns))

# Compare both constants with the median latency and count how many rows they over-predict
baseline_table = pd.DataFrame({
    "baseline": ["Dummy, raw target (arithmetic mean)", "Dummy, z (10^mean z = geometric mean)"],
    "prediction (ns)": [dummy_raw_value, dummy_z_value],
    "x median latency": [dummy_raw_value / median_latency, dummy_z_value / median_latency],
    "rows over-predicted": [(oof_ns["Dummy, raw target"] > latency_ns).mean(),
                            (oof_ns["Dummy, z"] > latency_ns).mean()],
})
print(baseline_table.to_string(index=False, float_format=lambda v: f"{v:12.4g}"))
print()
print(f"median measured latency   {median_latency:.4g} ns")

**(b) KNN.** With uniform weights the KNN averages the targets of its neighbours:
the arithmetic mean on the raw target and, back in ns, the geometric mean on $z$.
The arithmetic mean is always ≥ the geometric mean (equal only if all the values
are the same), so the raw-target prediction can only move up, towards the slowest
neighbour. The cell measures how different the two predictions really are.

In [ ]:
# --- (b) Raw-target KNN prediction divided by z KNN prediction, row by row ---
# A ratio of 1 means both KNNs predict the same latency for that row
ratio = oof_ns[f"KNN (K_NN={P23_NEIGHBOURS}), raw target"] / oof_ns[f"KNN (K_NN={P23_NEIGHBOURS}), z"]
print(f"raw-target / z prediction, same row, same folds")
print(f"  minimum            {ratio.min():.6f}")
print(f"  median             {np.median(ratio):.6f}")
print(f"  95th percentile    {np.quantile(ratio, 0.95):.4f}")
print(f"  maximum            {ratio.max():.4f}")
# Arithmetic mean >= geometric mean, so the ratio should never be below 1
print(f"  rows with ratio >= 1 (up to rounding)   {(ratio >= 1 - 1e-9).mean():.1%}")

Why are they so close? Each training configuration appears as five identical
feature vectors (its replicates). We refit the same $K_{NN} = 5$ pipeline on each
training fold and count how many different configurations the 5 neighbours of
each validation row come from. `config_group` is only used to read the answer; it
is never an input of the model.

In [ ]:
# --- How many configurations are among the 5 neighbours? ---
groups_array = storage[GROUP_COLUMN].to_numpy()     # only used to read the answer, never as input
configs_among_neighbours = np.empty(len(storage), dtype=int)
for train_idx, valid_idx in cv.split():
    # 1. Same pipeline as before, fitted on this fold's training rows
    fitted = make_knn(P23_NEIGHBOURS).fit(X.iloc[train_idx], z[train_idx])
    # 2. Transform the validation rows with the preprocessing fitted on the training rows
    X_valid_prepared = fitted.named_steps["prep"].transform(X.iloc[valid_idx])
    # 3. kneighbors returns, for each validation row, the positions of its 5 neighbours
    #    inside the training rows
    _, neighbour_pos = fitted.named_steps["knn"].kneighbors(X_valid_prepared)
    # 4. Translate those positions into configurations and count the different ones
    neighbour_groups = groups_array[train_idx][neighbour_pos]
    configs_among_neighbours[valid_idx] = [len(set(row)) for row in neighbour_groups]

# How many validation rows have 1, 2, ... different configurations among their neighbours
counts = pd.Series(configs_among_neighbours).value_counts().sort_index()
print("distinct configurations among the K_NN = 5 neighbours of a validation row")
print(pd.DataFrame({"configurations": counts.index, "validation rows": counts.to_numpy(),
                    "share": (counts / len(storage)).round(3).to_numpy()}).to_string(index=False))

**(c) Where each error measure puts its weight.** We split the rows into five
groups by their measured $z$. The first table gives the share of the total
absolute error in ns that falls in each group. The second gives the typical error
as a factor, $10^{\text{median}\,|z - \hat z|}$ (a factor, not ns).

In [ ]:
# --- (c) Errors by latency group ---
# Five groups of rows by their measured z (quintiles)
level = pd.qcut(z, 5, labels=["fastest 20%", "2nd", "3rd", "4th", "slowest 20%"])

share_rows, factor_rows = [], []
for name, pred_ns in oof_ns.items():
    abs_err_ns = np.abs(pred_ns - latency_ns)          # absolute error in ns
    abs_err_z = np.abs(z - np.log10(pred_ns))          # absolute error in z
    # Share of the total ns error that falls in each group
    share = pd.Series(abs_err_ns).groupby(level, observed=True).sum() / abs_err_ns.sum()
    # Typical error per group as a factor: 10^(median |z - z_hat|), never called ns
    factor = 10 ** pd.Series(abs_err_z).groupby(level, observed=True).median()
    share_rows.append({"model": name, **share.round(3).to_dict()})
    factor_rows.append({"model": name, **factor.round(2).to_dict()})

print("share of the total absolute error in ns, by latency group")
print(pd.DataFrame(share_rows).to_string(index=False))
print()
print("typical multiplicative error factor, by latency group (a factor, not ns)")
print(pd.DataFrame(factor_rows).to_string(index=False))

In [ ]:
# --- Bar charts of the two tables above, one bar per model in each group ---
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.0))
groups_order = ["fastest 20%", "2nd", "3rd", "4th", "slowest 20%"]
positions = np.arange(len(groups_order))
width = 0.2                                              # width of each bar
colours = ["#B0BEC5", "#023E83", "#F2C14E", "#1E7F4E"]   # one colour per model

for i, (rows, ax) in enumerate([(share_rows, axes[0]), (factor_rows, axes[1])]):
    for j, row in enumerate(rows):
        # Shift each model's bars so the four bars of a group sit side by side
        ax.bar(positions + (j - 1.5) * width, [row[g] for g in groups_order], width,
               color=colours[j], edgecolor="black", linewidth=0.5, label=row["model"])
    ax.set_xticks(positions)
    ax.set_xticklabels(groups_order, fontsize=8.5)
    ax.set_xlabel("validation rows grouped by measured latency")
    ax.spines[["top", "right"]].set_visible(False)

axes[0].set_ylabel("share of total |error| in ns")
axes[0].set_title("Where an error in nanoseconds puts its weight")
# Right panel in log scale, with plain numbers as tick labels
axes[1].set_yscale("log")
axes[1].yaxis.set_major_formatter(matplotlib.ticker.ScalarFormatter())
axes[1].yaxis.set_minor_formatter(matplotlib.ticker.NullFormatter())
axes[1].set_yticks([1, 2, 5, 10, 20, 50, 100])
axes[1].set_ylabel("typical error factor (log axis)")
axes[1].set_title("Typical multiplicative error per group")
axes[1].legend(fontsize=8, loc="upper center")
plt.tight_layout()
plt.show()

**What we see.**

- **(a)** The raw baseline predicts the arithmetic mean, about $2.7\times10^7$ ns,
  which is 23 times the median. It over-predicts 86% of the rows and, for the
  fastest fifth, it is wrong by a factor of about 80. The $z$ baseline predicts the
  geometric mean ($2.3\times10^6$ ns, twice the median), which is inside the bulk
  of the data.
- **(b)** With $K_{NN} = 5$, the 5 neighbours of every validation row are the 5
  replicates of one single training configuration (they share the same feature
  vector). So the prediction is just the average of the closest configuration's
  replicates, and the arithmetic and geometric means of 5 replicates only differ
  by the replicate noise (median ratio 1.0003, 95th percentile 1.03, worst case
  about 3). That is why both KNNs score the same. It happens because of this
  neighbourhood size: once the average mixes configurations with very different
  latencies, the arithmetic mean is pulled towards the slowest one.
- **(c)** For both KNNs, about 92% of the error in ns comes from the slowest fifth
  of the rows and only about 1% from the fastest three fifths. An error in ns
  basically only looks at the slow configurations: being 10 times wrong on a
  $3\times10^5$ ns configuration costs less than being 10% wrong on a
  $5\times10^7$ ns one. In $z$ the same factor costs the same at any level (a
  factor of 2 is always $\log_{10} 2 = 0.30$), and the KNN's typical factor stays
  between 1.3 and 1.8 in all five groups.

### 4.3 Why the modelling target is $z = \log_{10}(\texttt{latency\_ns})$

*This text goes unchanged into the memo and the defence pack.*

From here to Practice 5 the target for Challenge B is
$z = \log_{10}(\texttt{latency\_ns})$. The reasons are the scale of the data, how
errors are measured and the structure of the problem, not a better score: with
$K_{NN} = 5$ the KNN scores almost the same on both targets (section 4).

1. **Scale.** The latency covers about 4.3 orders of magnitude, with skewness ≈ 9
   and a mean 23 times the median. On that scale any average (the Dummy constant,
   a neighbour mean, a squared-error fit) is dominated by the few slowest
   configurations: the raw baseline ends up above 86% of the data. In $z$ the data
   cover about 4 units and the skewness is about 1.
2. **How errors are measured.** For latency what matters is the relative error:
   $10^6$ ns is nothing for a 2-second `hdd` write but a tenfold mistake for a
   sub-millisecond `nvme` read. Measured in ns, 92% of the error comes from the
   slowest fifth of the configurations. In $z$ the absolute error is a ratio,
   $|z - \hat z| = |\log_{10}(\widehat{\text{latency}} / \text{latency})|$, so the
   same factor costs the same at any level and every configuration counts. This is
   also why the course primary metric is MAE on $z$.
3. **Structure of the problem.** The predictors act as factors (device, access
   pattern, block size and queue depth multiply the time), and the replicate noise
   is mostly multiplicative: across levels its standard deviation grows about
   1,300 times in ns but only about 9 times in $z$. A multiplicative relation
   becomes additive on the log scale, which suits neighbour averages, linear
   models (P3) and squared-error training (P5). The log reduces the dependence of
   the noise on the level, but it does not remove it.
4. **Positive predictions.** A prediction on $z$ goes back to ns as $10^{\hat z}$,
   which is always finite and positive, as the submission requires. A model fitted
   on the raw target can predict a negative latency when it extrapolates (a linear
   model, for example).

**Rules that follow.** Log-scale metrics are computed from $z$ and $\hat z$.
Metrics in ns are computed only after $10^{\hat z}$. A log error is never turned
into ns: $10^{\text{MAE}(z)}$ can only be given as a multiplicative factor. The
submission always contains $10^{\hat z}$ in ns, never $z$.

## 5. P2.4: baseline and KNN on $z$

From here everything is on $z$, with R1's preprocessing and the canonical folds.
Before running anything we fix:

- **Baseline:** `DummyRegressor(strategy="mean")`, the same one R4 used for the
  comparison table.
- **Scaling:** one-hot on the 4 nominal predictors and `StandardScaler` on the 5
  numeric ones, fitted inside each fold (R1's block). We argue it in 5.1.
- **Grid:** $K_{NN} \in \{1, 5, 10, 15, 20, 25, 30, 40, 50, 75, 100\}$. In 4.2 we saw
  that the neighbours come in blocks of 5 (the replicates of one configuration),
  so apart from 1 we only use multiples of 5: $K_{NN} = 5m$ means "average of the
  $m$ closest configurations". We keep 1 and 5 to check this.
- **Selection rule:** the $K_{NN}$ with the lowest mean MAE(z) over the 5 folds; if
  two are exactly tied, the smaller one. The other KNN settings stay at their
  defaults (uniform weights, Euclidean distance).

All of this is chosen for storage only. Nothing is reused from the cyber model.

In [ ]:
# Candidate values of K_NN, fixed before any score (multiples of 5, plus 1)
N_NEIGHBOURS_GRID = [1, 5, 10, 15, 20, 25, 30, 40, 50, 75, 100]

# K_NN cannot be larger than the number of training rows in any fold
smallest_training_fold = min(len(train_idx) for train_idx, _ in cv.split())
print(f"smallest training fold   {smallest_training_fold} rows")
print(f"largest K_NN in the grid {max(N_NEIGHBOURS_GRID)}")
assert max(N_NEIGHBOURS_GRID) <= smallest_training_fold

### 5.1 Scaling decision

KNN uses the Euclidean distance, so a column with large numbers decides the
distance almost by itself (the hand version of this is Problem Sheet 1,
Exercise 1). For two random rows, the expected squared difference in a column is
$2\,\mathrm{Var}$, so the table shows each numeric predictor's share of the
squared distance before and after standardising. This is only a description of
the data: the scaler actually used is fitted inside each fold by the pipeline.

In [ ]:
# --- Weight of each numeric column in the Euclidean distance ---
# For two random rows, the expected squared difference in a column is 2 * variance
variance_raw = storage[NUMERIC_FEATURES].var(ddof=0)
share_raw = (2 * variance_raw) / (2 * variance_raw).sum()
# After StandardScaler every variance is 1, so the 5 columns weigh the same
share_scaled = pd.Series(1 / len(NUMERIC_FEATURES), index=NUMERIC_FEATURES)  # every variance becomes 1

scale_table = pd.DataFrame({
    "min": storage[NUMERIC_FEATURES].min(),
    "max": storage[NUMERIC_FEATURES].max(),
    "sd": np.sqrt(variance_raw).round(1),
    "share of distance, raw": share_raw.round(4),
    "share of distance, standardised": share_scaled.round(4),
})
print(scale_table.to_string())
print()
print("after standardising, each numeric column adds on average 2 to the squared distance;")
print("a different category in a one-hot column also adds 2 (one entry goes 1 -> 0, another 0 -> 1)")

**Decision.** Without scaling, `block_size` (from 4 to 1024) would be about 98% of
the distance, so the KNN would look almost only at the block size and ignore the
queue depth, the number of jobs or the read fraction. After `StandardScaler` every
numeric column has variance 1 in the training fold, so each one adds on average 2
to the squared distance, the same as a change of category in a one-hot column.
No column dominates just because of its units. We keep R1's block as it is and we
don't try an unscaled KNN as a candidate: the decision comes from how the distance
works, not from a score.

### 5.2 Baseline

In [ ]:
# --- Baseline on z, with R4's evaluation function ---
baseline_result = evaluate_storage(make_baseline(), X, z, latency_ns, cv)
baseline_folds = baseline_result["fold_metrics"]      # one row per fold
print(baseline_folds.round(4).to_string())
print()
print(f"MAE(z) mean {baseline_folds['mae_z'].mean():.4f}, "
      f"range {baseline_folds['mae_z'].min():.4f} - {baseline_folds['mae_z'].max():.4f}")

### 5.3 The $K_{NN}$ curve

Every candidate is scored with R4's `evaluate_storage` on the same canonical
folds. We report the whole curve, with the five fold values of each candidate,
not only the best point.

In [ ]:
# --- Score every K_NN of the grid on the canonical folds ---
grid_results = {}      # K_NN -> per-fold metrics
curve_rows = []        # one summary row per K_NN
for n_neighbours in N_NEIGHBOURS_GRID:
    fm = evaluate_storage(make_knn(n_neighbours), X, z, latency_ns, cv)["fold_metrics"]
    grid_results[n_neighbours] = fm
    curve_rows.append({
        "K_NN": n_neighbours,
        "configs averaged": n_neighbours / REPLICATES,   # neighbours come in blocks of 5 replicates
        **{f"f{q}": round(fm.loc[q, "mae_z"], 4) for q in fm.index},
        "mean MAE(z)": round(fm["mae_z"].mean(), 4),
        "sd": round(fm["mae_z"].std(ddof=0), 4),
        "range": round(fm["mae_z"].max() - fm["mae_z"].min(), 4),
        "RMSE(z)": round(fm["rmse_z"].mean(), 4),
        "R2(z)": round(fm["r2_z"].mean(), 4),
        "MAE_ns": f"{fm['mae_ns'].mean():.3e}",
    })
curve = pd.DataFrame(curve_rows)
print(curve.to_string(index=False))

# Selection rule declared in section 5: lowest mean MAE(z), ties -> smaller K_NN
# (the unrounded means are used, so a tie is a real tie and not a rounding effect)
exact_means = {k: grid_results[k]["mae_z"].mean() for k in N_NEIGHBOURS_GRID}
best_mean = min(exact_means.values())
SELECTED_NEIGHBOURS = min(k for k, v in exact_means.items() if v == best_mean)
print()
print(f"selected K_NN = {SELECTED_NEIGHBOURS} (mean MAE(z) = {best_mean:.6f})")

In [ ]:
# --- The whole K_NN curve with the fold-to-fold spread ---
fig, ax = plt.subplots(figsize=(8.2, 4.2))

# Grey points: the 5 fold values of each K_NN
for n_neighbours, fm in grid_results.items():
    ax.scatter([n_neighbours] * len(fm), fm["mae_z"], s=18, color="#B0BEC5", zorder=2)
# Mean line and min-max band over the folds
means = [grid_results[k]["mae_z"].mean() for k in N_NEIGHBOURS_GRID]
lows = [grid_results[k]["mae_z"].min() for k in N_NEIGHBOURS_GRID]
highs = [grid_results[k]["mae_z"].max() for k in N_NEIGHBOURS_GRID]
ax.fill_between(N_NEIGHBOURS_GRID, lows, highs, color="#DCE7F4", alpha=0.6, zorder=1,
                label="range over the 5 folds")
ax.plot(N_NEIGHBOURS_GRID, means, marker="o", color="#023E83", linewidth=2, zorder=3,
        label="mean over the 5 folds")
# Circle around the selected K_NN
ax.scatter([SELECTED_NEIGHBOURS], [exact_means[SELECTED_NEIGHBOURS]], s=150,
           facecolors="none", edgecolors="#1E7F4E", linewidths=2.2, zorder=4,
           label=f"selected, K_NN = {SELECTED_NEIGHBOURS}")

ax.set_xticks(N_NEIGHBOURS_GRID)
ax.set_xlabel("number of neighbours K_NN")
ax.set_ylabel("validation MAE(z)")
# The baseline (0.677) is far above the curve, so it is given in the title instead of drawn
ax.set_title("KNN on z, canonical folds (grey points: individual folds)\n"
             f"baseline MAE(z) = {baseline_folds['mae_z'].mean():.3f}, off the chart")
ax.legend(fontsize=8.5)
ax.grid(axis="y", linestyle=":", linewidth=0.6, alpha=0.7)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

**What we see.**

- $K_{NN} = 1$ and $K_{NN} = 5$ give almost the same MAE(z) (0.211 and 0.209). This
  matches 4.2: both only use the closest configuration (one replicate, or the mean
  of its 5 replicates).
- The big drop is from 5 to 10 (0.209 → 0.184), when the prediction starts to
  average two configurations instead of one. Relying on a single neighbouring
  configuration is noisy: the closest one probably differs from the query in some
  setting that matters a lot.
- The curve is flat between 15 and 25 (0.1801, 0.1800, 0.1809) and then goes up
  slowly. With more neighbours the average includes configurations that are not
  really similar, so the predictions get over-smoothed.
- Our rule picks $K_{NN} = 20$, but 15 and 20 differ by about 0.00003 in mean
  MAE(z), which is nothing compared with the fold range (0.008 to 0.011). On these
  folds we can't really tell 15, 20 and 25 apart. The rule was fixed before seeing
  the curve, so we keep 20.
- The best mean is a bit optimistic, because we chose it with the same 5 folds that
  measure it.
- MAE in ns is lowest at $K_{NN} = 10$, not at 20. It is the same point as in
  4.2 (c): a metric in ns mostly looks at the slow configurations, so it can
  prefer another model. We select with MAE(z), the primary metric.

### 5.4 Full metric set: baseline against the selected KNN

All metrics of the metric contract, per fold. MAE, RMSE and $R^2$ are on $z$;
MAE$_\text{ns}$ is computed after $10^{\hat z}$. The last block gives the paired
per-fold differences $\delta_f = \text{MAE}_{\text{baseline},f} - \text{MAE}_{\text{KNN},f}$,
so a positive value means the KNN was better on fold $f$.

In [ ]:
# --- Full metric set, per fold, for the baseline and the selected KNN ---
selected_folds = grid_results[SELECTED_NEIGHBOURS]
metric_names = ["mae_z", "rmse_z", "r2_z", "mae_ns", "fit_time_s"]

full_rows = []
for name, fm in [("Dummy (mean)", baseline_folds), (f"KNN, K_NN = {SELECTED_NEIGHBOURS}", selected_folds)]:
    for metric in metric_names:
        values = fm[metric]
        # One row per (model, metric): the 5 fold values, their mean and their range
        full_rows.append({"model": name, "metric": metric,
                          **{f"f{q}": values.loc[q] for q in values.index},
                          "mean": values.mean(), "range": values.max() - values.min()})
full_table = pd.DataFrame(full_rows)
print(full_table.to_string(index=False, float_format=lambda v: f"{v:10.4g}"))

# --- Paired per-fold differences (section 5.3 of the statement) ---
# MAE is an error, so baseline - KNN: a positive value means the KNN was better on that fold
delta = baseline_folds["mae_z"] - selected_folds["mae_z"]
print()
print("paired differences in MAE(z), baseline - KNN (positive = KNN better)")
print("  " + "  ".join(f"f{q}: {d:.4f}" for q, d in delta.items()))
print(f"  mean {delta.mean():.4f}, range {delta.min():.4f} - {delta.max():.4f}")
print()
# 10^MAE(z) is a multiplicative factor, never an error in ns (log-to-physical rule)
print(f"typical multiplicative error of the KNN, 10^MAE(z) = "
      f"{10 ** selected_folds['mae_z'].mean():.2f} (a factor, not nanoseconds)")

**What we see.** The KNN with $K_{NN} = 20$ is much better than the baseline in
every fold: MAE(z) 0.180 against 0.677. The paired differences go from 0.485 to
0.516, very similar in all five folds, so the improvement is much larger than the
fold-to-fold variation. $R^2(z)$ goes from about 0 (the baseline only predicts the
mean) to 0.91.

In ns the MAE goes from $2.7\times10^7$ to $1.7\times10^7$. In relative terms it
improves less, because this number depends almost only on the slowest
configurations. An MAE(z) of 0.18 means that typical predictions are within a
factor of about 1.5 of the measured latency ($10^{0.18} \approx 1.51$, a factor,
not ns).

Fitting is very cheap (about 0.02 s per fold) because KNN only stores the training
rows; the real cost is at prediction time, when it searches the neighbours.

### 5.5 Row for the comparison table

`comparativa.csv` belongs to R4, so we don't write it from here. We build our row
with the template columns and give it to R4 to append (R4 already added the
storage baseline row).

In [ ]:
# --- Our row for comparativa.csv (R4 appends it; we don't write R4's file) ---
# Column names read from the course template, so the row has exactly its format
template_columns = list(pd.read_csv("../../plantillas/comparativa.csv").columns)

knn_row = {
    "milestone": "P2", "challenge": "storage", "family": "KNN", "route": "primary",
    "configuration": f"n_neighbors={SELECTED_NEIGHBOURS}, weights=uniform, metric=euclidean; "
                     f"selected from {N_NEIGHBOURS_GRID} by lowest mean MAE(z)",
    "preprocessing": "one-hot on the 4 nominal predictors, StandardScaler on the 5 numeric ones, "
                     "fitted inside each fold; target z = log10(latency_ns)",
    "features": "features.md: the 9 distributed source predictors (frozen basis)",
    "validation": "canonical GroupKFold(5) on config_group, read from storage_folds.csv",
    "primary_metric_per_fold": " ".join(f"{v:.4f}" for v in selected_folds["mae_z"]),
    "primary_metric": round(selected_folds["mae_z"].mean(), 4),
    "primary_metric_spread": round(selected_folds["mae_z"].max() - selected_folds["mae_z"].min(), 4),
    "supporting_metrics": (f"RMSE(z) {selected_folds['rmse_z'].mean():.4f}; "
                           f"R2(z) {selected_folds['r2_z'].mean():.4f}; "
                           f"MAE_ns {selected_folds['mae_ns'].mean():.0f} (computed after inverting predictions)"),
    "fit_time_s": round(selected_folds["fit_time_s"].mean(), 4),
    "interpretability": "none (a prediction can only be explained by listing its neighbours)",
    "notes": "unlike the constant baseline it uses the configuration: it averages the "
             f"{SELECTED_NEIGHBOURS // REPLICATES} most similar training configurations",
}
# Same columns as the template, no more and no less
assert set(knn_row) == set(template_columns)
for column in template_columns:
    print(f"{column:<24} {knn_row[column]}")

## 6. P2.6: submission-format dry run (storage)

The goal is to produce a file in the **exact** final format and check it with
`validate_submission.py` now, instead of finding a problem at the single real
submission in P5. The final format is:

| Column | Content |
|---|---|
| `row_id` | exactly the `row_id` set of the feature table, each one once |
| `latency_ns` | the prediction **in ns** ($10^{\hat z}$, never $z$), finite and strictly positive |

**How we do it.** The final test will be configurations the model has never seen,
so we imitate that with canonical **fold 0**: we fit the selected pipeline
($K_{NN} = 20$) on folds 1 to 4 and predict fold 0. Fold 0 is only a format exercise:
we don't score it and it is not a pseudo-test set for later milestones. We also
write a feature table for these rows (`row_id` + the 9 predictors, **no target**),
because that is what the final-test file will look like and the validator needs it.

In [ ]:
# --- Dry-run setup ---
DRY_RUN_FOLD = 0                                         # canonical fold used as the "final test"
OUTPUT_DIR = Path("../../entregas_prueba")
DRY_RUN_FEATURES_PATH = OUTPUT_DIR / "storage_dry_run_features.csv"
DRY_RUN_SUBMISSION_PATH = OUTPUT_DIR / "storage_dry_run.csv"
# The header comes from the course template: row_id,latency_ns
SUBMISSION_HEADER = list(pd.read_csv(DATA_DIR / "storage_submission_template.csv").columns)

# Split the rows: folds 1-4 to fit the model, fold 0 to predict
is_dry_run = storage[FOLD_COLUMN].to_numpy() == DRY_RUN_FOLD
train_part = storage[~is_dry_run]
dry_run_part = storage[is_dry_run]

# The feature table of the subset: row_id + the 9 predictors, no target
dry_run_features = dry_run_part[[ID_COLUMN] + FEATURES]
OUTPUT_DIR.mkdir(exist_ok=True)                          # create the folder if it does not exist
dry_run_features.to_csv(DRY_RUN_FEATURES_PATH, index=False)

print(f"submission header (template)    {SUBMISSION_HEADER}")
print(f"training rows (folds 1-4)       {len(train_part)}")
print(f"dry-run rows (fold {DRY_RUN_FOLD})            {len(dry_run_part)}")
# Must be 0: the predicted configurations are never seen in training, like in the final test
print(f"configurations in both parts    "
      f"{len(set(train_part[GROUP_COLUMN]) & set(dry_run_part[GROUP_COLUMN]))}")
print(f"target in the feature table     {TARGET in dry_run_features.columns}")

### Fit, predict and go back to nanoseconds

The pipeline is fitted on $z$ of the training part only. The prediction $\hat z$ is
turned into ns with $10^{\hat z}$ before writing it. The magnitude check matters:
a file with $z$ values (numbers around 5 to 9) would also be positive and finite,
so the validator would accept it, but it would be a wrong submission.

In [ ]:
# --- Fit the selected pipeline on folds 1-4 (on z) and predict fold 0 ---
final_pipeline = make_knn(SELECTED_NEIGHBOURS)
final_pipeline.fit(train_part[FEATURES], np.log10(train_part[TARGET].to_numpy()))

z_hat = final_pipeline.predict(dry_run_features[FEATURES])   # predictions on the z scale
predicted_latency_ns = 10 ** z_hat                      # back to the physical scale

# Submission table with the exact header and column order of the template
submission = pd.DataFrame({ID_COLUMN: dry_run_features[ID_COLUMN].to_numpy(),
                           TARGET: predicted_latency_ns})
submission = submission[SUBMISSION_HEADER]

# Our own checks before writing the file
training_min = train_part[TARGET].min()
assert list(submission.columns) == SUBMISSION_HEADER                     # header and order
assert submission[ID_COLUMN].is_unique                                   # each row_id once
assert set(submission[ID_COLUMN]) == set(dry_run_features[ID_COLUMN])    # exactly the subset's rows
assert np.isfinite(submission[TARGET]).all() and (submission[TARGET] > 0).all()
# In ns, not z. A KNN averages training latencies, so it cannot go below the smallest one
assert submission[TARGET].min() >= training_min

submission.to_csv(DRY_RUN_SUBMISSION_PATH, index=False)

print(f"predicted z_hat range        {z_hat.min():.3f} to {z_hat.max():.3f}")
print(f"written latency_ns range     {submission[TARGET].min():.4g} to {submission[TARGET].max():.4g} ns")
print(f"smallest training latency    {training_min:.4g} ns")
print()
print(submission.head().to_string(index=False))

### Running the course validator

This is the same command as in the statement, run from `Proyecto/`. The validator
reads no target file and returns no score, only `valid` or a format error.

In [ ]:
import subprocess   # to run the course validator as an external command
import sys

PROJECT_DIR = Path("../..")      # the validator is run from Proyecto/, as in the statement


def run_validator(submission_path, features_path):
    # Build the command: python herramientas/validate_submission.py storage <submission> <features>
    # (paths written relative to Proyecto/)
    command = [sys.executable, "herramientas/validate_submission.py", "storage",
               str(Path(submission_path).resolve().relative_to(PROJECT_DIR.resolve())),
               str(Path(features_path).resolve().relative_to(PROJECT_DIR.resolve()))]
    # Run it and keep what it prints: "valid" or a format error
    result = subprocess.run(command, cwd=PROJECT_DIR, capture_output=True, text=True)
    return " ".join(command[1:]), (result.stdout + result.stderr).strip()


command, verdict = run_validator(DRY_RUN_SUBMISSION_PATH, DRY_RUN_FEATURES_PATH)
print(f"python3 {command}")
print(f"-> {verdict}")
assert verdict == "valid"

**Why our magnitude check is needed.** The cell below writes, in a temporary
folder, the same file but with $z$ instead of ns, and runs the validator on it.
Nothing in the project uses this file.

In [ ]:
import tempfile

# Temporary folder: it is deleted automatically when the "with" block ends
with tempfile.TemporaryDirectory(dir=OUTPUT_DIR) as tmp:
    wrong_path = Path(tmp) / "storage_with_z_instead_of_ns.csv"
    # Same file, but the latency column holds z_hat instead of 10^z_hat (a WRONG submission)
    submission.assign(**{TARGET: z_hat}).to_csv(wrong_path, index=False)
    _, wrong_verdict = run_validator(wrong_path, DRY_RUN_FEATURES_PATH)

print(f"validator on a file that contains z instead of ns -> {wrong_verdict}")

**What we see.** Our file passes the validator. The file with $z$ also passes,
even though it would be a wrong submission (its "latencies" would be around 6 ns).
The validator only checks the format, so the conversion $10^{\hat z}$ and the
magnitude check are our responsibility. In P5 we will repeat exactly these steps
with the released final-test feature table instead of fold 0.